# GRL Miniproject: Oversmoothing vs. Over-Squashing

**Research Question:** Do residual (skip) connections and graph rewiring complement each other
in addressing the oversmoothing/over-squashing trade-off, or does fixing one failure mode worsen the other?

**2×2 experimental grid:**

| | Original graph | Rewired graph (SDRF) |
|---|---|---|
| **GCN** | Baseline | Rewiring only |
| **SkipGCN** | Skip only | Skip + Rewiring |

Measured on:
- **Graph A** (oversmoothing): Stochastic Block Model — dense communities, label = community
- **Graph B** (over-squashing): Ring graph — label depends on node d hops away, vary d

## 0. Setup

In [ ]:
# Install dependencies (run once)
# !pip install -r requirements.txt

In [ ]:
import sys
sys.path.insert(0, 'src')

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from models import GCN, SkipGCN
from graphs import make_sbm_graph, make_ring_graph, make_barbell_graph, train_val_test_split
from rewiring import sdrf_rewire, curvature_stats
from metrics import mean_average_distance, dirichlet_energy, sensitivity_vs_distance
from training import train_model, evaluate

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# consistent color palette 
COLORS = {
    'GCN':          '#4878CF',
    'SkipGCN':      '#D65F5F',
    'GCN+Rewire':   '#6ACC65',
    'Skip+Rewire':  '#B47CC7',
}
LINESTYLES = {
    'GCN':          '-',
    'SkipGCN':      '--',
    'GCN+Rewire':   '-.',
    'Skip+Rewire':  ':',
}

## 1. Graph Construction

In [ ]:
# --- Graph A: SBM (oversmoothing target) ---
data_sbm = make_sbm_graph(
    n_communities=4, nodes_per_community=50,
    p_intra=0.7, p_inter=0.02, feature_dim=16, seed=SEED
)
data_sbm = train_val_test_split(data_sbm, seed=SEED)
print('SBM graph:', data_sbm)

# --- Graph B: Ring (over-squashing target) ---
# We'll vary target_distance for the squashing experiment;
# for the main 2x2 experiment use a fixed moderate distance
RING_D = 8
data_ring = make_ring_graph(n_nodes=64, feature_dim=8, target_distance=RING_D, seed=SEED)
data_ring = train_val_test_split(data_ring, seed=SEED)
print('Ring graph:', data_ring)

# --- Barbell (for curvature visualization) ---
data_barbell = make_barbell_graph(clique_size=15, bridge_length=5, feature_dim=8, seed=SEED)
data_barbell = train_val_test_split(data_barbell, seed=SEED)
print('Barbell graph:', data_barbell)

## 2. Graph Rewiring (SDRF)

We apply SDRF to create rewired versions of each graph.

In [ ]:
print('Computing curvature stats (original graphs)...')
stats_sbm_orig = curvature_stats(data_sbm)
stats_ring_orig = curvature_stats(data_ring)
stats_barbell_orig = curvature_stats(data_barbell)

print(f'SBM     — mean κ: {stats_sbm_orig["mean"]:+.3f}, min: {stats_sbm_orig["min"]:+.3f}')
print(f'Ring    — mean κ: {stats_ring_orig["mean"]:+.3f}, min: {stats_ring_orig["min"]:+.3f}')
print(f'Barbell — mean κ: {stats_barbell_orig["mean"]:+.3f}, min: {stats_barbell_orig["min"]:+.3f}')

In [ ]:
print('Rewiring SBM graph...')
data_sbm_rewired = sdrf_rewire(data_sbm, n_iterations=30, tau=0.0)
print(f'SBM edges: {data_sbm.edge_index.size(1)//2} -> {data_sbm_rewired.edge_index.size(1)//2}')

print('Rewiring Ring graph...')
data_ring_rewired = sdrf_rewire(data_ring, n_iterations=20, tau=0.0)
print(f'Ring edges: {data_ring.edge_index.size(1)//2} -> {data_ring_rewired.edge_index.size(1)//2}')

print('Rewiring Barbell graph...')
data_barbell_rewired = sdrf_rewire(data_barbell, n_iterations=20, tau=0.0)
print(f'Barbell edges: {data_barbell.edge_index.size(1)//2} -> {data_barbell_rewired.edge_index.size(1)//2}')

In [ ]:
# Curvature distribution: before vs after rewiring (Barbell — most illustrative)
from rewiring import compute_ricci_curvature

curv_before = list(compute_ricci_curvature(data_barbell).values())
curv_after  = list(compute_ricci_curvature(data_barbell_rewired).values())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, vals, title in zip(axes,
                            [curv_before, curv_after],
                            ['Before rewiring', 'After SDRF rewiring']):
    ax.hist(vals, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(0, color='tomato', lw=1.5, linestyle='--', label='κ = 0')
    ax.set_xlabel('Ollivier-Ricci curvature κ', fontsize=11)
    ax.set_ylabel('Edge count', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=10)

plt.suptitle('Curvature distribution — Barbell graph', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('curvature_distribution.pdf', bbox_inches='tight')
plt.show()

## 3. Experiment A: Oversmoothing on SBM Graph

Sweep depth K ∈ {2, 4, 8, 16} for all 4 conditions.
Metric: Mean Average Distance (MAD) of final node embeddings.

In [ ]:
LAYERS = [2, 4, 8, 16]
HID_DIM = 64
TRAIN_PARAMS = dict(lr=0.01, weight_decay=5e-4, epochs=300, patience=20, device=DEVICE)

conditions_sbm = {
    'GCN':         (GCN,     data_sbm),
    'SkipGCN':     (SkipGCN, data_sbm),
    'GCN+Rewire':  (GCN,     data_sbm_rewired),
    'Skip+Rewire': (SkipGCN, data_sbm_rewired),
}

results_mad = {name: [] for name in conditions_sbm}
results_de  = {name: [] for name in conditions_sbm}
results_acc_sbm = {name: [] for name in conditions_sbm}

input_dim = data_sbm.x.size(1)
n_classes = data_sbm.num_classes

for K in LAYERS:
    print(f'\n--- K={K} layers ---')
    for name, (ModelClass, graph) in conditions_sbm.items():
        torch.manual_seed(SEED)
        model = ModelClass(input_dim, HID_DIM, n_classes, K)
        model, _, _ = train_model(model, graph, **TRAIN_PARAMS)

        embeddings = model.embed(graph.x.to(DEVICE), graph.edge_index.to(DEVICE)).cpu()
        mad = mean_average_distance(embeddings)
        de  = dirichlet_energy(embeddings, graph.edge_index)
        acc = evaluate(model, graph, graph.test_mask, DEVICE)

        results_mad[name].append(mad)
        results_de[name].append(de)
        results_acc_sbm[name].append(acc)
        print(f'  {name:15s}  MAD={mad:.4f}  DE={de:.4f}  Test Acc={acc:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

metrics = [
    (results_mad,      'Mean Average Distance (MAD) ↑', 'MAD'),
    (results_de,       'Dirichlet Energy ↑',             'Energy'),
    (results_acc_sbm,  'Test Accuracy ↑',                'Accuracy'),
]

for ax, (data_dict, ylabel, title) in zip(axes, metrics):
    for name, vals in data_dict.items():
        ax.plot(LAYERS, vals, marker='o',
                label=name, color=COLORS[name], linestyle=LINESTYLES[name], lw=2)
    ax.set_xlabel('Number of layers K', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.set_xticks(LAYERS)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Experiment A: Oversmoothing on SBM graph', fontsize=13)
plt.tight_layout()
plt.savefig('exp_a_oversmoothing.pdf', bbox_inches='tight')
plt.show()

## 4. Experiment B: Over-Squashing on Ring Graph

### B.1 — Vary depth K (fixed target_distance=8)

In [ ]:
conditions_ring = {
    'GCN':         (GCN,     data_ring),
    'SkipGCN':     (SkipGCN, data_ring),
    'GCN+Rewire':  (GCN,     data_ring_rewired),
    'Skip+Rewire': (SkipGCN, data_ring_rewired),
}

results_acc_ring = {name: [] for name in conditions_ring}

input_dim_ring = data_ring.x.size(1)
n_classes_ring = data_ring.num_classes

for K in LAYERS:
    print(f'\n--- K={K} layers ---')
    for name, (ModelClass, graph) in conditions_ring.items():
        torch.manual_seed(SEED)
        model = ModelClass(input_dim_ring, HID_DIM, n_classes_ring, K)
        model, _, _ = train_model(model, graph, **TRAIN_PARAMS)
        acc = evaluate(model, graph, graph.test_mask, DEVICE)
        results_acc_ring[name].append(acc)
        print(f'  {name:15s}  Test Acc={acc:.3f}')

### B.2 — Vary target distance d (fixed K=8)

In [ ]:
DISTANCES = [2, 4, 6, 8, 10, 12]
FIXED_K = 8

results_acc_dist = {name: [] for name in ['GCN', 'SkipGCN', 'GCN+Rewire', 'Skip+Rewire']}

for d in DISTANCES:
    print(f'\n--- target_distance={d} ---')
    g_orig   = train_val_test_split(make_ring_graph(64, 8, d, SEED), seed=SEED)
    g_rewire = sdrf_rewire(g_orig, n_iterations=20, tau=0.0)

    for name, (ModelClass, graph) in [
        ('GCN',         (GCN,     g_orig)),
        ('SkipGCN',     (SkipGCN, g_orig)),
        ('GCN+Rewire',  (GCN,     g_rewire)),
        ('Skip+Rewire', (SkipGCN, g_rewire)),
    ]:
        torch.manual_seed(SEED)
        model = ModelClass(g_orig.x.size(1), HID_DIM, g_orig.num_classes, FIXED_K)
        model, _, _ = train_model(model, graph, **TRAIN_PARAMS)
        acc = evaluate(model, graph, graph.test_mask, DEVICE)
        results_acc_dist[name].append(acc)
        print(f'  {name:15s}  Acc={acc:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# B.1: accuracy vs depth
ax = axes[0]
for name, vals in results_acc_ring.items():
    ax.plot(LAYERS, vals, marker='o',
            label=name, color=COLORS[name], linestyle=LINESTYLES[name], lw=2)
ax.axvline(RING_D, color='gray', lw=1, linestyle=':', alpha=0.7, label=f'K=d={RING_D}')
ax.set_xlabel('Number of layers K', fontsize=11)
ax.set_ylabel('Test Accuracy ↑', fontsize=11)
ax.set_title(f'Acc vs Depth  (d={RING_D})', fontsize=12)
ax.set_xticks(LAYERS)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# B.2: accuracy vs required range
ax = axes[1]
for name, vals in results_acc_dist.items():
    ax.plot(DISTANCES, vals, marker='o',
            label=name, color=COLORS[name], linestyle=LINESTYLES[name], lw=2)
ax.axvline(FIXED_K, color='gray', lw=1, linestyle=':', alpha=0.7, label=f'K={FIXED_K}')
ax.set_xlabel('Required range d (hops)', fontsize=11)
ax.set_ylabel('Test Accuracy ↑', fontsize=11)
ax.set_title(f'Acc vs Required Range  (K={FIXED_K})', fontsize=12)
ax.set_xticks(DISTANCES)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Experiment B: Over-Squashing on Ring graph', fontsize=13)
plt.tight_layout()
plt.savefig('exp_b_oversquashing.pdf', bbox_inches='tight')
plt.show()

## 5. Experiment C: Jacobian Sensitivity vs Distance

Measures how much influence a source node has on a distant target node.
Decay rate = proxy for over-squashing severity.

In [ ]:
JACOBIAN_K = 8  # fixed depth for Jacobian experiment
jacobian_results = {}

for name, (ModelClass, graph) in [
    ('GCN',         (GCN,     data_ring)),
    ('SkipGCN',     (SkipGCN, data_ring)),
    ('GCN+Rewire',  (GCN,     data_ring_rewired)),
    ('Skip+Rewire', (SkipGCN, data_ring_rewired)),
]:
    torch.manual_seed(SEED)
    model = ModelClass(data_ring.x.size(1), HID_DIM, data_ring.num_classes, JACOBIAN_K)
    model, _, _ = train_model(model, graph, **TRAIN_PARAMS)

    sens = sensitivity_vs_distance(model, graph, n_pairs=30, max_distance=12, device=DEVICE)
    jacobian_results[name] = sens
    print(f'{name}: {sens}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for name, sens_dict in jacobian_results.items():
    dists = sorted(sens_dict.keys())
    vals  = [sens_dict[d] for d in dists]
    ax.plot(dists, vals, marker='o',
            label=name, color=COLORS[name], linestyle=LINESTYLES[name], lw=2)

ax.set_xlabel('Graph distance between node pair', fontsize=11)
ax.set_ylabel('Mean Jacobian norm  ||∂h_v / ∂x_u||', fontsize=11)
ax.set_title(f'Jacobian sensitivity vs distance  (K={JACOBIAN_K})', fontsize=12)
ax.set_yscale('log')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('exp_c_jacobian.pdf', bbox_inches='tight')
plt.show()

## 6. Summary Table

In [ ]:
import pandas as pd

# Report at K=8 (middle ground)
K_IDX = LAYERS.index(8)

rows = []
for name in ['GCN', 'SkipGCN', 'GCN+Rewire', 'Skip+Rewire']:
    rows.append({
        'Model': name,
        'MAD (SBM) ↑':       f"{results_mad[name][K_IDX]:.4f}",
        'Acc SBM ↑':         f"{results_acc_sbm[name][K_IDX]:.3f}",
        'Acc Ring (d=8) ↑':  f"{results_acc_ring[name][K_IDX]:.3f}",
    })

df = pd.DataFrame(rows).set_index('Model')
print(df.to_string())
df